In [0]:
%python
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    IntegerType
)

pipeline_name = "retail_data_pipeline"

try:
    customers = spark.table("default.top_customers")
    products = spark.table("default.top_products")
    revenue = spark.table("default.monthly_revenue")

    customer_count = customers.count()
    product_count = products.count()
    revenue_count = revenue.count()

    status = "SUCCESS"
    error_message = None

except Exception as e:
    customer_count = 0
    product_count = 0
    revenue_count = 0
    status = "FAILED"
    error_message = str(e)

log_data = [
    Row(
        pipeline_name=pipeline_name,
        run_timestamp=datetime.now(),
        status=status,
        customer_rows=customer_count,
        product_rows=product_count,
        revenue_rows=revenue_count,
        error_message=error_message
    )
]

schema = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("run_timestamp", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("customer_rows", IntegerType(), False),
    StructField("product_rows", IntegerType(), False),
    StructField("revenue_rows", IntegerType(), False),
    StructField("error_message", StringType(), True)
])

log_df = spark.createDataFrame(log_data, schema)

log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("default.retail_pipeline_logs")

display(log_df)